# 최종 모델 — 성별 분리 + Poly(deg 2) + Ridge + Round

## 최종 성능
- **OOF RMSE**: 0.1337
- **리더보드**: 0.12382

## 핵심 전략
1. **성별 분리**: 남/여 칼로리 소모 공식이 다르므로 각각 별도 모델 학습
2. **핵심 5변수만 사용**: Exercise_Duration, BPM, Temp_diff, Age, Weight(lb)
   - 불필요한 파생변수를 넣으면 Polynomial 조합 폭발로 분산 증가
3. **Polynomial(degree=2) + Ridge**: 곱(상호작용) 구조 학습
   - degree=3보다 degree=2가 성별 분리 시 더 안정적
4. **정수 반올림**: 타겟이 정수이므로 round된 값이 실제 지표에 직접 유리
5. **OOF 기반 그리드 탐색**: degree × alpha 조합을 성별별로 탐색

## Final Model Summary
| 항목 | 남성 (M) | 여성 (F) |
|------|---------|---------|
| Degree | 2 | 2 |
| Alpha | 1e-06 | 0.000763 |
| OOF RMSE | 0.1370 | 0.1303 |

---
## 1. 라이브러리 임포트

In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

---
## 2. 데이터 로드

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sub = pd.read_csv('sample_submission.csv')

print(f"Train: {train.shape}, Test: {test.shape}")

---
## 3. 피처 엔지니어링 — Temp_diff만 추가

이전 실험에서 23개 파생변수를 만들었지만, 성별 분리 + Polynomial이면
원본 변수 5개만으로 충분하다는 것을 확인

Temp_diff만 추가하는 이유:
- 체온은 "절대값"보다 정상 체온(98.6°F)에서 얼마나 벗어났는지가 운동 강도와 더 직접 연결
- 단순한 상수 뺄셈이라 데이터 누수 위험 없음

In [ ]:
def add_features(df):
    """Temp_diff 파생변수 추가: 정상 체온(98.6°F) 기준 편차"""
    df = df.copy()
    df['Temp_diff'] = df['Body_Temperature(F)'] - 98.6
    # 예: 체온 101.6°F → Temp_diff = 3.0 (정상보다 3도 높음)
    return df

train = add_features(train)
test = add_features(test)

# 핵심 5변수: OOF 기반 제거 실험을 반복한 결과 이 조합이 최적
COLS = ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)']
y = train['Calories_Burned'].values.astype(float)

# 성별 마스크
m_tr = (train['Gender'] == 'M').values  # train 남성 인덱스
f_tr = ~m_tr                             # train 여성 인덱스
m_te = (test['Gender'] == 'M').values   # test 남성 인덱스
f_te = ~m_te                             # test 여성 인덱스

print(f"핵심 피처: {COLS}")
print(f"남성 train/test: {m_tr.sum()}/{m_te.sum()}")
print(f"여성 train/test: {f_tr.sum()}/{f_te.sum()}")

---
## 4. 유틸리티 함수

- `round_clip`: 예측값을 정수로 반올림 + 음수 방지
- `oof_predict`: KFold OOF 예측 (fold마다 poly/scaler를 독립적으로 fit → 누수 완전 차단)

In [ ]:
def round_clip(pred):
    """예측값을 정수로 반올림하고 음수를 0으로 클리핑
    
    타겟(Calories_Burned)이 정수이므로 반올림이 실제 평가 지표에 직접 유리
    np.rint: numpy 배열 전체에 한번에 적용 (벡터 연산)
    """
    return np.rint(np.clip(pred, 0, None))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def oof_predict(X, y, degree, alpha):
    """KFold OOF 예측 — 각 fold에서 poly/scaler/모델을 독립적으로 fit
    
    핵심: validation fold는 transform만 수행하여 데이터 누수를 완전 차단
    """
    oof = np.zeros(len(y))
    for tr_idx, va_idx in kf.split(X):
        # include_bias=False: Ridge가 자체적으로 intercept를 학습하므로 상수항 불필요
        poly = PolynomialFeatures(degree=degree, include_bias=False)
        sc = StandardScaler()  # 평균 0, 표준편차 1로 정규화
        mdl = Ridge(alpha=alpha, random_state=42)
        
        # train fold: fit_transform (fit과 transform 동시 수행)
        Xtr = sc.fit_transform(poly.fit_transform(X[tr_idx]))
        # validation fold: train에서 fit된 변환기로 transform만 (누수 방지!)
        Xva = sc.transform(poly.transform(X[va_idx]))
        
        mdl.fit(Xtr, y[tr_idx])
        oof[va_idx] = mdl.predict(Xva)
    return oof

---
## 5. 성별별 (degree, alpha) 그리드 탐색

**탐색 전략**:
- Degree: [2, 3, 4, 5] — 성별 분리 시 degree 2가 충분할 수 있음
- Alpha: logspace(1e-6 ~ 10, 18개) — L2 규제 강도
- 평가: OOF 예측을 **정수 반올림한 후** RMSE (실제 제출 지표와 동일한 기준)

남/여 각각 독립적으로 최적 조합을 찾아 성별별 특성을 최대한 반영

In [ ]:
# 탐색 공간 정의
degrees = [2, 3, 4, 5]            # 다항식 차수 후보
alphas = np.logspace(-6, 1, 18)   # 규제 강도 후보 18개 (1e-6 ~ 10)

best = {}  # 성별별 최적 (degree, alpha) 저장

for gender, mask in [('M', m_tr), ('F', f_tr)]:
    Xg = train.loc[mask, COLS].values.astype(float)  # 해당 성별 데이터 추출
    yg = y[mask]
    
    best_score, best_deg, best_alpha = 1e18, None, None
    
    for deg in degrees:
        for alpha in alphas:
            # OOF 예측 후 반올림 → round RMSE 직접 최적화
            score = rmse(yg, round_clip(oof_predict(Xg, yg, deg, alpha)))
            if score < best_score:
                best_score, best_deg, best_alpha = score, deg, alpha
    
    best[gender] = (best_deg, best_alpha)
    print(f"[{gender}] degree={best_deg} | alpha={best_alpha:.4g} | OOF RMSE={best_score:.6f}")

---
## 6. 최적 파라미터로 전체 OOF RMSE 확인

In [ ]:
# 남성+여성 합산 OOF 생성
oof_all = np.zeros(len(train))

for gender, mask in [('M', m_tr), ('F', f_tr)]:
    deg, alpha = best[gender]
    oof_all[mask] = oof_predict(
        train.loc[mask, COLS].values.astype(float),
        y[mask], deg, alpha
    )

# 전체 OOF RMSE (round 적용)
total_rmse = rmse(y, round_clip(oof_all))
print(f"\n 전체 OOF RMSE: {total_rmse:.6f}")

---
## 7. 테스트 예측 (전체 train으로 재학습)

OOF 탐색에서 찾은 최적 파라미터를 사용하되, 
train 전체를 학습에 활용해 모델 성능을 최대화

In [ ]:
def full_predict(X_tr, y_tr, X_te, degree, alpha):
    """전체 train으로 학습 후 test 예측"""
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    sc = StandardScaler()
    mdl = Ridge(alpha=alpha, random_state=42)
    
    # train 전체로 fit_transform → 모델 학습
    mdl.fit(sc.fit_transform(poly.fit_transform(X_tr)), y_tr)
    # test는 train에서 fit된 변환기로 transform만 (누수 방지)
    return mdl.predict(sc.transform(poly.transform(X_te)))

# 성별별 test 예측
test_pred = np.zeros(len(test))

for gender, tr_mask, te_mask in [('M', m_tr, m_te), ('F', f_tr, f_te)]:
    deg, alpha = best[gender]
    test_pred[te_mask] = full_predict(
        train.loc[tr_mask, COLS].values.astype(float), y[tr_mask],
        test.loc[te_mask, COLS].values.astype(float),
        deg, alpha
    )

print(f"Test 예측 통계:")
print(f"  mean: {test_pred.mean():.2f}")
print(f"  min:  {test_pred.min():.2f}")
print(f"  max:  {test_pred.max():.2f}")

---
## 8. 제출 파일 생성

In [ ]:
# 정수 반올림 후 저장
sub['Calories_Burned'] = round_clip(test_pred).astype(int)
sub.to_csv('submit_gender_best.csv', index=False)

print("\n저장 완료: submit_gender_best.csv")
print(f"제출 통계: mean={sub['Calories_Burned'].mean():.2f}, "
      f"min={sub['Calories_Burned'].min()}, max={sub['Calories_Burned'].max()}")

---
##  Final Model Summary

### 모델 구성
- **피처**: ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)']
- **성별 분리**: M/F 각각 독립 모델
- **Polynomial Degree**: M=2, F=2
- **Ridge Alpha**: M=1e-06, F=0.000763
- **후처리**: 정수 반올림 (np.rint + clip)

### 성능
- **OOF RMSE**: 0.1337 (M: 0.1370, F: 0.1303)
- **리더보드**: 0.12382

### 왜 이 모델이 최적인가?
1. **성별 분리**: 남/여 칼로리 소모 공식이 물리적으로 다름 → 분리 학습이 유리
2. **핵심 변수만 사용**: 불필요한 피처는 Polynomial 조합 폭발로 분산만 증가
3. **Degree 2로 충분**: 성별 분리 시 2차 상호작용만으로 곱셈 구조를 포착
4. **Round RMSE 직접 최적화**: 정수 타겟 특성을 탐색 단계부터 반영
5. **OOF 기반**: 누수 없는 일반화 성능으로 판단

---
##  전체 프로젝트 성능 추이

| # | 단계 | 모델 | OOF RMSE | 핵심 개선 |
|---|------|------|----------|----------|
| 1 | Baseline | LGBM | 2.241 | 4모델 비교 |
| 2 | + Log transform | LGBM | 1.993 | 분포 보정 |
| 3 | + 파생변수 | LGBM | 1.775 | Intensity, Effort |
| 4 | + Optuna | LGBM | ~1.37 | 하이퍼파라미터 |
| 5 | Poly(deg 3) | Ridge | 0.920 | **모델 전환** |
| 6 | + Ridge + KFold | Ridge | ~0.54 | 규제 + 안정화 |
| 7 | + 타겟 변환 스태킹 | Stacking | 0.318 | 다양화 앙상블 |
| 8 | + 가중치 최적화 | Stacking | 0.309 | scipy |
| 9 | + Feature pruning | Ridge | 0.300 | 노이즈 제거 |
| 10 | + Round | Ridge | 0.221 | 정수 타겟 |
| 11 | + 성별 분리 | Ridge | 0.157 | M/F 분리 |
| 12 | **+ 그리드 탐색** | **Ridge** | **0.134** | **degree/alpha 최적화** |

**RMSE 2.24 → 0.13: 총 94% 감소** 

---
## 시각화 자료

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ---------------------------
# Config
# ---------------------------
CSV_PATH = "results/model_comparison.csv"
OUT_DIR = "results/plots"

# Phase boundaries (based on your project write-up / figures)
# Phase 1: Tree (Stage 1-4)
# Phase 2: Polynomial (Stage 5-10)
# Phase 3: Gender (Stage 11-12)
PHASE_RANGES = {
    "Tree Models": (1, 4),
    "Polynomial Regression": (5, 10),
    "Gender Split": (11, 12),
}

# Colors (matched to your figure vibe)
COLOR_TREE = "#ff6b6b"     # coral-ish red
COLOR_POLY = "#33c9c8"     # teal
COLOR_GENDER = "#ffd54f"   # warm yellow

SHADE_TREE = "#ffe5e5"
SHADE_POLY = "#e6fbfb"
SHADE_GENDER = "#fff3d6"


# ---------------------------
# Utils
# ---------------------------
def ensure_outdir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def fmt_stage_label(model_name: str) -> str:
    """
    Try to wrap labels similarly to your screenshot.
    e.g. "+ Log transform" -> "+ Log\ntransform"
    """
    # Manual small heuristic line-breaks
    replacements = {
        "Log transform": "Log\ntransform",
        "Feature Engineering": "Feature\nEngineering",
        "Optuna Tuning": "Optuna\nTuning",
        "Poly(deg3) Ridge": "Poly(deg3)\nRidge",
        "Ridge + KFold": "+ Ridge\n+ KFold",
        "Target Transform Stacking": "+ Target\nStacking",
        "Weight Optimization": "+ Weight\nOptimize",
        "Feature Pruning": "+ Feature\nPruning",
        "Round (integer)": "+ Round\n(Integer)",
        "Gender Split": "+ Gender\nSplit",
        "Grid Search (Final)": "+ Grid\nSearch",
        "LGBM (baseline)": "Baseline\nLGBM",
    }
    return replacements.get(model_name, model_name)

def load_df(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    required = {"Stage", "Model", "RMSE", "Improvement_pct", "Phase", "Key_Technique"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"model_comparison.csv is missing columns: {missing}")
    df = df.sort_values("Stage").reset_index(drop=True)
    return df

def stage_to_phase(stage: int) -> str:
    for name, (s, e) in PHASE_RANGES.items():
        if s <= stage <= e:
            return name
    return "Unknown"


# ---------------------------
# Plot 1: Stage-by-stage improvement bar chart
# ---------------------------
def plot_improvement_by_stage(df: pd.DataFrame, out_path: str) -> None:
    # Improvement_pct: Stage 1 is baseline; often 0.0 there
    x = np.arange(len(df))
    improvements = df["Improvement_pct"].astype(float).values

    # Colors by phase, but keep early bars lighter like your figure
    colors = []
    for stage in df["Stage"].astype(int).values:
        ph = stage_to_phase(stage)
        if ph == "Tree Models":
            colors.append("#89e7b0")  # light green for early baseline steps
        elif ph == "Polynomial Regression":
            colors.append("#27ae60")  # stronger green
        elif ph == "Gender Split":
            colors.append("#8ee9b7")  # light green again
        else:
            colors.append("#cccccc")

    plt.figure(figsize=(22, 7))
    bars = plt.bar(x, improvements, color=colors, edgecolor="#2c3e50", linewidth=1.0)

    # Value labels on bars
    for rect, val in zip(bars, improvements):
        if np.isnan(val):
            continue
        plt.text(
            rect.get_x() + rect.get_width() / 2,
            rect.get_height() + 0.6,
            f"{int(round(val))}%",
            ha="center",
            va="bottom",
            fontsize=12,
            color="#2c3e50",
            fontweight="bold"
        )

    # X tick labels
    labels = [fmt_stage_label(m) for m in df["Model"].tolist()]
    plt.xticks(x, labels, fontsize=11)
    plt.ylabel("Improvement (%)", fontsize=16, fontweight="bold")
    plt.title("Stage-by-Stage RMSE Improvement — Key Turning Points", fontsize=22, fontweight="bold")

    # Vertical separators between phases (after stage 4 and after stage 10)
    # We use x-index positions corresponding to stage transitions
    # find idx where stage == 4 and stage == 10
    idx_stage4 = df.index[df["Stage"] == 4].tolist()
    idx_stage10 = df.index[df["Stage"] == 10].tolist()

    for idx in idx_stage4:
        plt.axvline(idx + 0.5, color="#95a5a6", linestyle="--", linewidth=1.5, alpha=0.8)
    for idx in idx_stage10:
        plt.axvline(idx + 0.5, color="#95a5a6", linestyle="--", linewidth=1.5, alpha=0.8)

    # Phase text labels (approx positions like your screenshot)
    ymax = max(45, np.nanmax(improvements) + 5)
    plt.ylim(0, ymax)

    def center_idx(stage_start: int, stage_end: int) -> float:
        # convert stage to df index range
        i0 = df.index[df["Stage"] == stage_start][0]
        i1 = df.index[df["Stage"] == stage_end][0]
        return (i0 + i1) / 2

    plt.text(center_idx(1, 4), ymax * 0.92, "Phase 1\nTree Models",
             ha="center", va="top", fontsize=13, color=COLOR_TREE, fontweight="bold")
    plt.text(center_idx(5, 10), ymax * 0.92, "Phase 2\nPolynomial",
             ha="center", va="top", fontsize=13, color=COLOR_POLY, fontweight="bold")
    plt.text(center_idx(11, 12), ymax * 0.92, "Phase 3\nGender",
             ha="center", va="top", fontsize=13, color=COLOR_GENDER, fontweight="bold")

    plt.grid(axis="y", linestyle="-", alpha=0.25)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


# ---------------------------
# Plot 2: RMSE reduction by phase + contribution pie
# ---------------------------
def plot_phase_summary(df: pd.DataFrame, out_path: str) -> None:
    # Compute phase endpoints based on your phase definition
    # Start RMSE = stage 1 RMSE
    start_rmse = float(df.loc[df["Stage"] == 1, "RMSE"].iloc[0])

    # End RMSE of each phase = RMSE at last stage in that phase
    phase_end_rmse = {}
    for ph_name, (s, e) in PHASE_RANGES.items():
        phase_end_rmse[ph_name] = float(df.loc[df["Stage"] == e, "RMSE"].iloc[0])

    end_tree = phase_end_rmse["Tree Models"]
    end_poly = phase_end_rmse["Polynomial Regression"]
    end_gender = phase_end_rmse["Gender Split"]

    # Reductions (negative values, like in your figure text)
    red_tree = end_tree - start_rmse
    red_poly = end_poly - end_tree
    red_gender = end_gender - end_poly
    total_red = end_gender - start_rmse

    # For pie contributions, use absolute reduction magnitudes
    contrib_vals = np.array([abs(red_tree), abs(red_poly), abs(red_gender)], dtype=float)
    contrib_labels = [
        f"Phase 1: Tree\n({red_tree:+.3f})",
        f"Phase 2: Poly\n({red_poly:+.3f})",
        f"Phase 3: Gender\n({red_gender:+.3f})",
    ]
    contrib_colors = [COLOR_TREE, COLOR_POLY, COLOR_GENDER]

    # Figure layout: left bar-style phase reductions, right pie chart
    fig = plt.figure(figsize=(20, 7))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.05, 1])

    # ---- Left: "RMSE Reduction by Phase" as blocks (visual like your screenshot)
    ax1 = fig.add_subplot(gs[0, 0])

    # We draw rectangles from phase start RMSE to phase end RMSE for each phase
    phases = [
        ("Phase 1\nTree Models\n(Stage 1-4)", start_rmse, end_tree, COLOR_TREE),
        ("Phase 2\nPolynomial\n(Stage 5-10)", end_tree, end_poly, COLOR_POLY),
        ("Phase 3\nGender Split\n(Stage 11-12)", end_poly, end_gender, COLOR_GENDER),
    ]

    x_positions = np.arange(len(phases))
    width = 0.7

    for i, (label, y0, y1, c) in enumerate(phases):
        # rectangle from min(y0,y1) to max, but y1 < y0 (rmse decreases)
        top = y0
        bottom = y1
        height = top - bottom
        ax1.bar(i, height, bottom=bottom, width=width, color=c, alpha=0.75, edgecolor="#2c3e50", linewidth=1.5)

        reduction = y1 - y0  # negative
        pct = (abs(reduction) / abs(total_red)) * 100 if total_red != 0 else 0

        ax1.text(i, bottom + height/2,
                 f"{reduction:+.3f}\n({pct:.0f}%)",
                 ha="center", va="center", fontsize=14, fontweight="bold", color="#2c3e50")

        # annotate endpoints
        ax1.text(i, y0 + 0.03, f"{y0:.3f}", ha="center", va="bottom", fontsize=10, color="#7f8c8d")
        ax1.text(i, y1 - 0.03, f"{y1:.3f}", ha="center", va="top", fontsize=10, color="#7f8c8d")

    ax1.set_xticks(x_positions)
    ax1.set_xticklabels([p[0] for p in phases], fontsize=11)
    ax1.set_ylabel("RMSE", fontsize=16, fontweight="bold")
    ax1.set_title("RMSE Reduction by Phase", fontsize=20, fontweight="bold")
    ax1.set_ylim(0, max(start_rmse + 0.25, 2.5))
    ax1.grid(axis="y", linestyle="-", alpha=0.25)

    # ---- Right: contribution pie
    ax2 = fig.add_subplot(gs[0, 1])
    wedges, texts, autotexts = ax2.pie(
        contrib_vals,
        labels=contrib_labels,
        colors=contrib_colors,
        startangle=90,
        autopct=lambda p: f"{p:.1f}%",
        textprops={"fontsize": 12},
        wedgeprops={"edgecolor": "white", "linewidth": 2},
        explode=(0.02, 0.04, 0.06),
    )
    for at in autotexts:
        at.set_fontweight("bold")
        at.set_color("black")
        at.set_fontsize(14)

    ax2.set_title(f"Contribution to Total RMSE Reduction ({start_rmse:.2f} -> {end_gender:.2f})",
                  fontsize=18, fontweight="bold")

    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


# ---------------------------
# Plot 3: RMSE progression line plot with phase shading + annotations
# ---------------------------
def plot_rmse_progression(df: pd.DataFrame, out_path: str) -> None:
    stages = df["Stage"].astype(int).values
    rmse = df["RMSE"].astype(float).values
    labels = [fmt_stage_label(m) for m in df["Model"].tolist()]

    start_rmse = rmse[0]
    end_rmse = rmse[-1]
    improvement_pct_total = (1 - end_rmse / start_rmse) * 100 if start_rmse != 0 else 0

    plt.figure(figsize=(22, 7))
    ax = plt.gca()

    # Phase shading by stage span (convert to x-index ranges)
    # x-axis is categorical indices 0..n-1
    x = np.arange(len(df))

    def shade_phase(stage_start: int, stage_end: int, color: str):
        i0 = df.index[df["Stage"] == stage_start][0]
        i1 = df.index[df["Stage"] == stage_end][0]
        ax.axvspan(i0 - 0.5, i1 + 0.5, color=color, alpha=0.6)

    shade_phase(1, 4, SHADE_TREE)
    shade_phase(5, 10, SHADE_POLY)
    shade_phase(11, 12, SHADE_GENDER)

    # Line + points
    ax.plot(x, rmse, color="#2c3e50", linewidth=3)
    ax.scatter(x, rmse, s=160, color="#ff6b6b", edgecolor="#2c3e50", linewidth=2, zorder=3)

    # recolor points by phase (like your screenshot has phase-colored markers)
    for i, st in enumerate(stages):
        ph = stage_to_phase(int(st))
        if ph == "Tree Models":
            ax.scatter([i], [rmse[i]], s=180, color=COLOR_TREE, edgecolor="#2c3e50", linewidth=2, zorder=4)
        elif ph == "Polynomial Regression":
            ax.scatter([i], [rmse[i]], s=180, color=COLOR_POLY, edgecolor="#2c3e50", linewidth=2, zorder=4)
        elif ph == "Gender Split":
            ax.scatter([i], [rmse[i]], s=180, color=COLOR_GENDER, edgecolor="#2c3e50", linewidth=2, zorder=4)

    # Value labels on points
    for i, val in enumerate(rmse):
        ax.text(i, val + 0.05, f"{val:.3f}", ha="center", va="bottom",
                fontsize=11, color="#2c3e50", fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylabel("RMSE", fontsize=16, fontweight="bold")
    ax.set_title(f"Calorie Prediction — RMSE Progression ({start_rmse:.2f} → {end_rmse:.2f}, -{improvement_pct_total:.0f}%)",
                 fontsize=22, fontweight="bold")
    ax.set_ylim(0, max(2.5, start_rmse + 0.25))
    ax.grid(axis="y", linestyle="-", alpha=0.25)

    # Legend (phase shading)
    legend_handles = [
        plt.Rectangle((0, 0), 1, 1, color=SHADE_TREE, alpha=0.6, label="Phase 1: Tree Models"),
        plt.Rectangle((0, 0), 1, 1, color=SHADE_POLY, alpha=0.6, label="Phase 2: Polynomial Regression"),
        plt.Rectangle((0, 0), 1, 1, color=SHADE_GENDER, alpha=0.6, label="Phase 3: Gender Split"),
    ]
    ax.legend(handles=legend_handles, loc="upper right", fontsize=12)

    # Annotations: model switch & gender breakthrough (positions aligned to your plot)
    # Model switch at stage 5 (Poly introduced)
    idx_poly = df.index[df["Stage"] == 5][0]
    ax.annotate(
        "Model Switch!\nTree -> Poly",
        xy=(idx_poly, rmse[idx_poly]),
        xytext=(idx_poly, rmse[idx_poly] + 0.65),
        arrowprops=dict(arrowstyle="->", color=COLOR_TREE, linewidth=2),
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
        color=COLOR_TREE,
    )

    # Gender split at stage 11
    idx_gender = df.index[df["Stage"] == 11][0]
    ax.annotate(
        "Gender Split\nBreakthrough!",
        xy=(idx_gender, rmse[idx_gender]),
        xytext=(idx_gender, rmse[idx_gender] + 0.35),
        arrowprops=dict(arrowstyle="->", color=COLOR_TREE, linewidth=2),
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
        color=COLOR_TREE,
    )

    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


# ---------------------------
# Main
# ---------------------------
def main():
    ensure_outdir(OUT_DIR)
    df = load_df(CSV_PATH)

    plot_improvement_by_stage(df, os.path.join(OUT_DIR, "improvement_by_stage.png"))
    plot_phase_summary(df, os.path.join(OUT_DIR, "phase_summary.png"))
    plot_rmse_progression(df, os.path.join(OUT_DIR, "rmse_progression.png"))

    print("[OK] Saved plots to:", OUT_DIR)

if __name__ == "__main__":
    main()